# مُرشِّحُ التردداتِ المنخفضةِ (Low-Pass Filter)

**مجموعةُ البياناتِ**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**القنواتُ**: P4, Cz, F8, T7  
**معدّلُ أخذِ العيناتِ**: 200 Hz  
**المُشاركُ**: 1 (مُشاركٌ واحدٌ فقط للسرعةِ)

---

## ماذا يَعمَلُ هذا الدفترُ؟

يُزيلُ مُرشِّحُ التردداتِ المنخفضةِ المكوّناتِ ذاتَ الترددِ العاليِّ (فوقَ 40 Hz) من إشارةِ EEG. ويشملُ ذلك:

- تدخّلَ خطِّ الكهرباءِ (50/60 Hz)
- شوائبَ العضلاتِ (EMG، وهي عاليةُ الترددِ)
- الضجيجَ العشوائيَّ عاليَ الترددِ

نُطبِّقُ هذا المُرشِّحَ **بعدَ** مُرشِّحِ التردداتِ العاليةِ، فتكونُ الإشارةُ خاليةً من إزاحةِ التيارِ والانجرافِ.

## ما الذي ينبغي أن تتوقّعَهُ؟

بعدَ الترشيحِ، تُصبحُ الإشارةُ **أكثرَ نعومةٍ** — تختفي التذبذباتُ السريعةُ المتعرّجةُ من ناتجِ المُرشِّحِ العاليِّ. تحتفظُ الإشارةُ بالتردداتِ ذاتِ الصلةِ بالدماغِ (1 إلى 40 Hz: دلتا، ثيتا، ألفا، بيتا).

## المُعاملاتُ الرئيسةُ

| المُعاملُ | القيمةُ | المعنى |
| --------- | ------- | ------ |
| ترددُ القطعِ | 40 Hz | تُزالُ التردداتُ فوقَ 40 Hz |
| الترتيبُ | 4 | شدّةُ انحدارِ المُرشِّحِ |
| الطريقةُ | Butterworth | استجابةٌ مُسطّحةٌ في نطاقِ التمريرِ |
| filtfilt | نعم | صفريُّ الطورِ (لا تأخيرَ زمنيًّا) |

## 1. تثبيتُ المكتباتِ

In [ ]:
!pip install scipy numpy plotly wfdb

## 2. استنساخُ المستودعِ وتنزيلُ مُشاركٍ واحدٍ

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')

In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1

## 3. تحميلُ الإشارةِ وتطبيقُ المُرشِّحِ العاليِّ أوّلًا

نُطبِّقُ مُرشِّحَ التردداتِ العاليةِ (1 Hz) أوّلًا لإزالةِ الانجرافِ، ثمَّ نُطبِّقُ مُرشِّحَ التردداتِ المنخفضةِ (40 Hz) لإزالةِ الضجيجِ. هاتان الخطوتانِ تُكافئانِ مُرشِّحَ نطاقٍ، الذي سنراهُ في الدفترِ التاليِ.

In [ ]:
import numpy as np
from scipy import signal
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
channel_data = eeg_data[:, 0]  # P4 channel
fs = 200  # Sampling rate (Hz)

# Step 1: High-pass filter at 1 Hz
def butter_highpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = signal.butter(order, normal_cutoff, btype='high', analog=False)
    return signal.filtfilt(b, a, data)

# Step 2: Low-pass filter at 40 Hz
def butter_lowpass_filter(data, cutoff, fs, order=4):
    nyq = 0.5 * fs
    normal_cutoff = cutoff / nyq
    b, a = signal.butter(order, normal_cutoff, btype='low', analog=False)
    return signal.filtfilt(b, a, data)

filtered_hp = butter_highpass_filter(channel_data, cutoff=1.0, fs=fs)
filtered_lp = butter_lowpass_filter(filtered_hp, cutoff=40.0, fs=fs)

print(f'Channels: {ch_names}')
print(f'Signal length: {len(channel_data)} samples ({len(channel_data)/fs:.1f} seconds)')

## 4. رسمٌ تفاعليٌّ: خامٌ، ثمَّ عاليٌّ، ثمَّ منخفضٌ

تُظهرُ ثلاثةُ رسومٍ فرعيةٍ التطوّرَ الكاملَ:

1. **EEG الخامُ** (أعلى) — انجرافٌ + ضجيجٌ
2. **بعدَ المُرشِّحِ العاليِّ** (وسط) — أُزيلَ الانجرافُ، بقيَ الضجيجُ
3. **بعدَ المُرشِّحِ المنخفضِ** (أسفل) — أُزيلَ الانجرافُ والضجيجُ معًا

**ما الذي تبحثُ عنهُ؟**
- الإشارةُ الخامُ فيها انجرافٌ بطيءٌ (ميلٌ تدريجيٌّ)
- إشارةُ المُرشِّحِ العاليِّ مُتمركزةٌ حولَ الصفرِ لكنّها متعرّجةٌ
- إشارةُ المُرشِّحِ المنخفضِ مُتمركزةٌ حولَ الصفرِ **وناعمةٌ**
- التذبذباتُ ذاتُ الصلةِ بالدماغِ (الموجاتُ البطيئةُ) محفوظةٌ في الثلاثةِ جميعًا

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

n_plot = min(5000, len(channel_data))
t_sec = timestamps[:n_plot] / 1000.0

fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=('Raw EEG (P4)', 'After high-pass (1 Hz)', 'After low-pass (40 Hz)'))

fig.add_trace(go.Scatter(x=t_sec, y=channel_data[:n_plot],
                         name='Raw', line=dict(color='gray', width=0.5)),
               row=1, col=1)
fig.add_trace(go.Scatter(x=t_sec, y=filtered_hp[:n_plot],
                         name='High-pass', line=dict(color='green', width=0.5)),
               row=2, col=1)
fig.add_trace(go.Scatter(x=t_sec, y=filtered_lp[:n_plot],
                         name='HP + LP', line=dict(color='red', width=0.5)),
               row=3, col=1)

fig.update_layout(height=800, title_text='Low-Pass Filter: Raw -> High-pass -> Low-pass',
                  xaxis3_title='Time (s)', yaxis_title='EEG (uV)',
                  yaxis2_title='EEG (uV)', yaxis3_title='EEG (uV)')
fig.show()

## 5. ماذا تعلّمنا؟

- أزالَ مُرشِّحُ التردداتِ المنخفضةِ عندَ 40 Hz **الضجيجَ عاليَ الترددِ** من الإشارةِ
- أصبحتِ الإشارةُ **أكثرَ نعومةٍ** مع الحفاظِ على التردداتِ ذاتِ الصلةِ بالدماغِ (1 إلى 40 Hz)
- مُجتمعةً مع المُرشِّحِ العاليِّ، حصلنا على إشارةٍ نظيفةٍ في **النطاقِ 1 إلى 40 Hz**
- يُظهرُ الدفترُ التاليُ كيفيةَ القيامِ بذلك في **خطوةٍ واحدةٍ** باستخدامِ مُرشِّحِ النطاقِ